## Introduction

#### This notebook implements the sentiment analysis system using a transformer-based model. It includes data handling, inference, and interface logic

#### This is a GUI for sentiment analysis using HuggingFace's transformer model.

#### This provides a UI for the user to perform sentiment analysis, either through text input or data from a CSV files. Utilises the cardiffnlp/twitter-roberta-base-sentiment-latest model from HuggingFace to make prediction of the sentiments and then displays the results.

#### After displaying the result, it provides means for user to save the new results locally as a .CSV file, if provided through the file processing tab

## Library Imports

#### This section loads all required libraries and dependencies.

In [31]:
# Import required libraries
import tkinter as tk
from tkinter import *
from tkinter import ttk, filedialog

import pandas as pd
import numpy as np
import seaborn as sns
import json

import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger_eng')
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from scipy.special import softmax
from tqdm.notebook import tqdm

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


### Load sentiment analysis model from Hugging Face

In [32]:
MODEL = f"cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Window, title assignment, and size definition

In [33]:
window = Tk()
window.title("Sentiment Analysis GUI")
window.geometry("1000x650")

''

### Tab Layout definition

In [34]:
style = ttk.Style()
style.configure("TNotebook.Tab", padding=[20, 10])

tab_control = ttk.Notebook(window)
tab1 = ttk.Frame(tab_control)
tab2 = ttk.Frame(tab_control)
tab3 = ttk.Frame(tab_control)

### Notebook tabs

In [35]:
tab_control.add(tab1, text="Text Analysis")
tab_control.add(tab2, text="File Processor")
tab_control.add(tab3, text="Help")

tab_control.pack(expand=1,fill='both')

### Text Analysis Tab

In [36]:
label1 = Label(tab1,text="Use here to analyse individual reviews", font=("Comic Sans MS",12), padx=10,pady=5)
label2 = Label(tab2,text="Use here to analyse CSV/JSON/JSONL/TXT files", font=("Comic Sans MS",12), padx=5,pady=5)
label3 = Label(tab3,text="Instructions on how to use the app", font=("Comic Sans MS",12), padx=5,pady=5)

label1.grid(column=0, row=0)
label2.grid(column=0, row=0,columnspan=2,sticky="w")
label3.grid(column=0, row=0)

#### Perform tokenization, POS tagging, Sentiment analysis of text/sentences inputted into the textbox. Also cleans/clear both inputted texts as well as results.

#### Retrieves text entered into the input widget, tokenizes it, performs POS tagging, processes it through the sentiment analysis model, and displays the predicted sentiment.

#### Updates the result in the GUI's output label

### Tokenisation function

In [37]:
def get_tokens():
    global raw_text
    raw_text = str(raw_entry.get())
    global tokens
    tokens = tokenizer.tokenize(raw_text)
    result = '\nTokens: {}'.format(tokens)
    # Insert into display
    tab1_display.insert(tk.END,result)

### POS Tagging function

In [38]:
def get_pos_tags():
    pos_tags = nltk.pos_tag(tokens)
    result = '\nPOS Tagger: {}'.format(pos_tags)
    # Insert into display
    tab1_display.insert(tk.END,result)

### Sentiment analysis function

In [39]:
def get_sentiment():
    raw_text = tab1_display.get("1.0",tk.END)
    encoded_text = tokenizer(raw_text, return_tensors="pt")
    output = model(**encoded_text)
    scores = output[0][0].detach().numpy()
    scores = softmax(scores)
    scores_dict = {
        "Negative" : scores[0],
        "Neutral" : scores[1],
        "Positive" : scores[2]
    }
    for key, value in scores_dict.items():
        tab1_display.insert(tk.END, f"\n{key}: {value:.4f}\n")

### Clear input text field

In [40]:
def clear_text_entry():
    entry1.delete(0,END)

### Clear display result

In [41]:
def clear_display_result():
    tab1_display.delete('1.0',END)

In [42]:
l1 = Label(tab1,text="Enter Text to Analyse", font=("Comic Sans MS", 14))
l1.grid(row=3, column=0)
          
raw_entry = StringVar()
entry1 = Entry(tab1, textvariable=raw_entry,width=50, font=("Arial", 14))
entry1.grid(row=3,column=1, pady=10)

### Buttons definition

In [43]:
button1 = Button(tab1,text="Tokenize",width=15,bg="cornflower blue",fg="snow",command=get_tokens)
button1.grid(row=6,column=0, 
             padx=2,pady=10)
button1.config(font=("Comic Sans MS", 15))

button2 = Button(tab1,text="POS Tagger",width=15,bg="cornflower blue",fg="snow",command=get_pos_tags)
button2.grid(row=6,column=1, 
             padx=(0, 250),pady=10)
button2.config(font=("Comic Sans MS", 15))

button3 = Button(tab1,text="Sentiment",width=15,bg="cornflower blue",fg="snow",command=get_sentiment)
button3.grid(row=6,column=1, 
             padx=2,pady=10,
            sticky="e")
button3.config(font=("Comic Sans MS", 15))

button4 = Button(tab1,text="Clear Entry Text",width=15,bg="cornflower blue",fg="snow",command=clear_text_entry)
button4.grid(row=7,column=0, 
             padx=2,pady=10)
button4.config(font=("Comic Sans MS", 15))

button5 = Button(tab1,text="Clear Result",width=15,bg="cornflower blue",fg="snow",command=clear_display_result)
button5.grid(row=7,column=1, 
             padx=2,pady=10, 
             sticky="e")
button5.config(font=("Comic Sans MS", 15))

### Result screen display

In [44]:
tab1_display = Text(tab1, height=10, width=80)
tab1_display.grid(row=8, column=0, columnspan=4,
                  padx=5, pady=5,
                  sticky="nsew")

tab1.rowconfigure(8, weight=1)

for i in range(4):
    tab1.columnconfigure(i, weight=1)

## Tab 2 File Processor

#### Perform sentiment analysis on data from selected CSV/JSON/JSONL/TXT file.
#### Once the user has selected a file, the data is read, processed using the sentiment analysis model, result is displayed in a scrollable widget as negative, neutral, and positive where they add up to 1.
#### It performs sentiment analysis on the first 50 rows/key and this can be edited depending on compute power.

### Supported File Formats:
- #### CSV: Reads text from the review/text column
- #### JSON: Reads text from the review/text key
- #### JSONL: Reads text from the review/text key
- #### TXT: Reads text from document

### Tree view for .csv file

In [45]:
tree = ttk.Treeview(tab2)
tree.grid(row=3, column=0, columnspan=3, sticky="nsew")

# Allow resizing
tab2.rowconfigure(3, weight=1)
for i in range(3):
    tab2.columnconfigure(i, weight=1)

### File opening function 

#### Checks if the file is a CSV/JSON/JSONL/TXT files and loads the file accordingly

In [46]:
def open_files():
    global df
    global limited_json
    global limited_jsonl
    global txts
    global file_path
    
    file_path = tk.filedialog.askopenfilename(filetypes=[("CSV Files","*.csv"),
                                                         ("JSON Files","*.json"),
                                                         ("JSONL Files","*.jsonl"),
                                                         ("Text Files","*.txt")])
    if file_path:
        try:
            # Clear existing content
            displayed_file.delete("1.0", "end")
            
            if file_path.endswith(".csv"):
                df = pd.read_csv(file_path, encoding="latin-1", index_col=0)
                
                # Convert DataFrame to string and insert
                displayed_file.insert("end", "Preview (first 50 rows):\n\n")
                displayed_file.insert("end",
                                      df.head(500).to_string(index=False, col_space=0))
                
            elif file_path.endswith(".json"):
                with open(file_path, "r", encoding="utf-8") as file:
                    texts = json.load(file) 
                limited_json = texts[:500]
        
                # Improved format: JSON
                formatted = json.dumps(limited_json, indent=4)
                
                # Insert formatted JSON to display
                displayed_file.delete("1.0", tk.END)
                displayed_file.insert("end", "Preview (first 50 reviews):\n\n")
                displayed_file.insert(tk.END, formatted)
                
            elif file_path.endswith(".jsonl"):
                limited_jsonl = []
                
                with open(file_path, "r") as file:
                    for i, line in enumerate(file):
                        if i >= 500:
                            break
                        limited_jsonl.append(json.loads(line))

                # Improved format: JSONL 
                formatted = json.dumps(limited_jsonl, indent=4)

                # Insert formatted JSONL to display
                displayed_file.delete("1.0", tk.END)
                displayed_file.insert("end", "Preview (first 50 reviews):\n\n")
                displayed_file.insert(tk.END, formatted)
            
            elif file_path.endswith(".txt"):
                txts = []

                with open(file_path, "r", encoding="utf-8") as file:
                    for i, line in enumerate(file):
                        line = line.strip()
                        # skip empty lines
                        if line:
                            txts.append({"Id": str(i), "review": line})
                        if i >= 499:
                            break

                df = pd.DataFrame(txts)

                # Display text preview
                displayed_file.delete("1.0", tk.END)
                displayed_file.insert(tk.END, "Preview (first 50 rows):\n\n")
                displayed_file.insert(tk.END, df.head(500).to_string())
                
        except Exception as e:
            displayed_file.delete("1.0", tk.END)
            displayed_file.insert(tk.END, f"Error in processing the file: {e}")

In [47]:
l1 = Label(tab2,text="Insert CSV/JSON/JSONL/TXT file for processing", font=("Comic Sans MS", 14))
l1.grid(row=2, column=1, columnspan=2)

### Merged files save function

In [48]:
def save_file(final_df):
    file_path = filedialog.asksaveasfilename(
        defaultextension=".csv",
        filetypes=[
            ("CSV files", "*.csv"),
            ("JSONL files", "*.jsonl"),
            ("JSON files", "*.json"),
            ("TXT files", "*.txt")
        ]
    )

    if not file_path:
        return

    if file_path.endswith(".csv"):
        final_df.to_csv(file_path, index=False)

    elif file_path.endswith(".txt"):
        final_df.to_excel(file_path, index=False)

    elif file_path.endswith(".json"):
        final_df.to_json(file_path, orient="records", indent=4)

    elif file_path.endswith(".jsonl"):
        final_df.to_json(file_path, orient="records", indent=4)

    print("File saved successfully!")

### Sentiment analysis helper function

In [49]:
df = None
def get_file_sentiment(text):
    encoded = tokenizer(text, padding="max_length", truncation=True, max_length=512, return_tensors="pt")
    output = model(**encoded)
    scores = output[0][0].detach().numpy()
    scores = softmax(scores)
    return {"Negative": scores[0], "Neutral": scores[1], "Positive": scores[2]}

### Sentiment analysis function

In [50]:
def run_sentiment_analysis():
    output_text = ""
    results = {}
    global df
    # Setting the dataframe
    if file_path.endswith(".csv"):
        if df is None:
            print("Error: No CSV loaded.")
            return
        limited_df = df.head(50)
        
    elif file_path.endswith(".json"):
        if limited_json is None:
            print("Error: No JSON loaded.")
            return
        limited_df = pd.DataFrame(limited_json).head(50).copy()

    elif file_path.endswith(".jsonl"):
        if limited_jsonl is None:
            print("Error: No JSONL loaded.")
            return
        limited_df = pd.DataFrame(limited_jsonl).head(50).copy()

    elif file_path.endswith(".txt"):
        if txts is None:
            print("Error: No txt loaded.")
            return
        limited_df = pd.DataFrame(txts).head(50).copy()
        
    else:
        return
        
    # Prioritize 'review'/'text' and 'id'/'Id'
    text_col = 'review' if 'review' in limited_df.columns else ('text' if 'text' in limited_df.columns else None)
    id_col = 'id' if 'id' in limited_df.columns else ('Id' if 'Id' in limited_df.columns else None)

    if not text_col:
        print("Error: Could not find review/text column.")
        return

    # Perform Sentiment Analysis
    results = {}
    tab2_display_text.delete("1.0", "end")
    for i, row in tqdm(limited_df.iterrows(), total=len(limited_df)):
        text = row[text_col]
        myid = str(row[id_col]) if id_col else str(i)
        
        scores = get_file_sentiment(text)
        results[myid] = scores
        
        # UI Update
        for key, value in scores.items():    
            tab2_display_text.insert(tk.END, f"\n{key}: {value:.4f}\n")

    # Merge results
    results_df = pd.DataFrame.from_dict(results, orient="index").reset_index()
    results_df = results_df.rename(columns={"index": "Id"})
    
    # Ensure ID types match for merging
    if id_col:
        limited_df[id_col] = limited_df[id_col].astype(str)
        final_df = pd.merge(limited_df, results_df, left_on=id_col, right_on="Id", how="left")
    else:
        # If no ID, join by index
        limited_df['Id'] = limited_df.index.astype(str)
        final_df = pd.merge(limited_df, results_df, on="Id", how="left")
    
    # save file
    save_file(final_df)

    # Display merged file on console
    if final_df is not None:
        print(final_df.head()) # Prints to console
        
    else:
        print("Analysis failed or returned empty.")
        
    return final_df

### Clear file display

In [51]:
def clear_text_file():
    displayed_file.delete('1.0',END)

### Clear result display

In [52]:
def clear_result():
    tab2_display_text.delete('1.0',END)

### File display

In [53]:
displayed_file = Text(tab2, height=10)
displayed_file.grid(row=3,column=0,columnspan=4, 
                    padx=5,
                    sticky="nsew")
tab2.rowconfigure(3, weight=0)
for i in range(4):
    tab2.columnconfigure(i, weight=1)

# scroll functionality
scroll = tk.Scrollbar(tab2, command=displayed_file.yview)
scroll.grid(row=3, column=4, sticky="ns")

displayed_file.config(yscrollcommand=scroll.set)

### Result Display

In [54]:
tab2_display_text = Text(tab2,height=20)
tab2_display_text.grid(row=6,column=0,columnspan=4,
                       padx=5,
                       sticky="sew")
tab2.rowconfigure(6, weight=0)
for i in range(4):
    tab2.columnconfigure(i, weight=1)

# scroll functionality
scroll = tk.Scrollbar(tab2, command=displayed_file.yview)
scroll.grid(row=6, column=4, sticky="ns")

tab2_display_text.config(yscrollcommand=scroll.set)

### Open file button

In [55]:
button_tab_1 = Button(tab2,text="Open File",width=20,bg="green2",fg="snow",command=open_files)
button_tab_1.grid(row=4,column=0, 
             padx=10,pady=20,
             sticky="w")
button_tab_1.config(font=("Comic Sans MS", 15))

### Get file sentiment button

In [56]:
button_tab_2 = Button(tab2,text="Get File Sentiment",width=20,bg="green2",fg="snow",command=run_sentiment_analysis)
button_tab_2.grid(row=4,column=1, 
             padx=10,pady=20,
             sticky="w")
button_tab_2.config(font=("Comic Sans MS", 15))

### Clear display file button

In [57]:
button_tab_3 = Button(tab2,text="Clear File",width=20,bg="green2",fg="snow",command=clear_text_file)
button_tab_3.grid(row=4,column=2, 
             padx=10,pady=20,
             sticky="w")
button_tab_3.config(font=("Comic Sans MS", 15))

### Clear result button

In [58]:
button_tab_4 = Button(tab2,text="Clear Result",width=20,bg="green2",fg="snow",command=clear_result)
button_tab_4.grid(row=4,column=3, 
             padx=10,pady=20,
             sticky="w")
button_tab_4.config(font=("Comic Sans MS", 15))

## Help Tab

In [59]:
Help_info_1 = Label(tab3,text="Performs sentiment analysis on the first 50 rows, this can be changed depending on preference.",
                  font=("Comic Sans MS",12),padx=5,pady=5)
Help_info_2 = Label(tab3,text="Ensure the file format is '.csv/.json/.jsonl/.txt'.",
                  font=("Comic Sans MS",12),padx=5,pady=5)
Help_info_3 = Label(tab3,text="Ensure there is an 'id' tab for joining to occur.",
                  font=("Comic Sans MS",12),padx=5,pady=5)
Help_info_4 = Label(tab3,text="Ensure the file review/text column/key has the review on which sentiment analysis is to be performed.",
                  font=("Comic Sans MS",12),padx=5,pady=5)
Help_info_1.grid(column=0,row=2,sticky="w")
Help_info_2.grid(column=0,row=3,sticky="w")
Help_info_3.grid(column=0,row=4,sticky="w")
Help_info_4.grid(column=0,row=5,sticky="w")

### Windows mainloop 

In [60]:
window.mainloop()